# Differential Equations — Session 8
## Section 2.5: Solutions by Substitution

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. use the chain rule to transform a first-order equation.
2. recognize a homogeneous first-order equation and use $y=ux$.
3. solve a Bernoulli equation with $u=y^{1-n}$.
4. reduce $y'=F(Ax+By+C)$ to a separable equation.
5. reverse the substitution and check the resulting solution.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Topic |
|---:|---|
| 0–12 min | Substitution as structure recognition |
| 12–38 min | Homogeneous equations |
| 38–65 min | Bernoulli equations |
| 65–82 min | Linear-combination substitution |
| 82–90 min | Method selection and exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import erf
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def slope_field(f, xlim=(-3, 3), ylim=(-3, 3), density=21, title=None):
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    S = np.asarray(f(X, Y), dtype=float)
    S = np.nan_to_num(S, nan=0.0, posinf=20.0, neginf=-20.0)
    U = np.ones_like(S)
    length = np.sqrt(U**2 + S**2)
    plt.quiver(X, Y, U/length, S/length, angles="xy", pivot="mid")
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.xlabel("x")
    plt.ylabel("y")
    if title:
        plt.title(title)

def euler_method(f, x0, y0, h, n_steps):
    xs = np.empty(n_steps + 1)
    ys = np.empty(n_steps + 1)
    xs[0], ys[0] = x0, y0
    for n in range(n_steps):
        ys[n+1] = ys[n] + h*f(xs[n], ys[n])
        xs[n+1] = xs[n] + h
    return xs, ys

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 2.5-A — Homogeneous function

A function $F(x,y)$ is **homogeneous of degree $m$** if

$$
F(tx,ty)=t^mF(x,y)
$$

for every admissible $t$.

A first-order differential equation

$$
M(x,y)\,dx+N(x,y)\,dy=0
$$

is called homogeneous when $M$ and $N$ are homogeneous of the same degree.

### Theorem 2.5-B — Homogeneous substitution

If an equation can be written as

$$
y'=F\left(\frac{y}{x}\right),
$$

then the substitution

$$
y=ux
$$

transforms it into a separable equation because

$$
y'=u+xu'.
$$

### Definition 2.5-C — Bernoulli equation

A **Bernoulli equation** has the form

$$
y'+P(x)y=Q(x)y^n,
\qquad n\ne0,1.
$$

### Theorem 2.5-D — Bernoulli linearization

The substitution

$$
u=y^{1-n}
$$

transforms the Bernoulli equation into the linear equation

$$
u'+(1-n)P(x)u=(1-n)Q(x).
$$

Any excluded solution, especially $y=0$, must be checked separately.

### Proposition 2.5-E — Linear-combination substitution

If

$$
y'=F(Ax+By+C),
\qquad B\ne0,
$$

then

$$
u=Ax+By+C
$$

produces an autonomous equation

$$
u'=A+B F(u),
$$

which is separable.

### Classroom Checkpoint — Bernoulli Substitution

For

$$
y'+P(x)y=Q(x)y^n,\qquad n\ne0,1,
$$

which substitution converts the equation to a linear equation?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. General substitution idea

Suppose

$$
y=g(x,u(x)).
$$

Then the chain rule gives

$$
\frac{dy}{dx}=g_x+g_u\frac{du}{dx}.
$$

The goal is to choose $u$ so the transformed equation becomes separable or linear.

## 2. Homogeneous first-order equations

A normal-form equation

$$
y'=F\left(\frac{y}{x}\right)
$$

suggests

$$
u=\frac{y}{x},
\qquad y=ux,
\qquad y'=u+xu'.
$$

### Example

Solve

$$
y'=\frac{x+y}{x-y}.
$$

Substitute $y=ux$:

$$
u+xu'=\frac{1+u}{1-u}.
$$

Therefore

$$
xu'=\frac{1+u^2}{1-u}.
$$

Separate:

$$
\frac{1-u}{1+u^2}\,du=\frac{dx}{x}.
$$

Integrating,

$$
\arctan u-\frac12\ln(1+u^2)=\ln|x|+C.
$$

Finally, replace $u$ by $y/x$.

In [ ]:
# Visualize numerical solutions and rays of constant u=y/x
def homogeneous_rhs(x, y):
    return (x+y[0])/(x-y[0])

for y0 in [-1.5, -0.5, 0.5, 1.5]:
    sol = solve_ivp(homogeneous_rhs, (1, 4), [y0],
                    t_eval=np.linspace(1, 4, 500),
                    events=lambda x, y: x-y[0])
    plt.plot(sol.t, sol.y[0], label=fr"$y(1)={y0}$")

x_ray = np.linspace(0, 4, 100)
for u in [-1, -0.5, 0.5, 1]:
    plt.plot(x_ray, u*x_ray, linestyle="--", alpha=0.6)

plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Solutions and rays of constant ratio $u=y/x$")
plt.legend()
plt.show()

## 3. Bernoulli equations

A Bernoulli equation has the form

$$
y'+P(x)y=Q(x)y^n,
\qquad n\ne0,1.
$$

Use

$$
u=y^{1-n}.
$$

Then

$$
u'=(1-n)y^{-n}y',
$$

and the transformed equation is linear:

$$
u'+(1-n)P(x)u=(1-n)Q(x).
$$

### Example

Solve

$$
y'+y=xy^2.
$$

Here $n=2$, so set

$$
u=y^{-1}.
$$

The transformed equation is

$$
u'-u=-x.
$$

Solving the linear equation gives

$$
u=x+1+Ce^x.
$$

Therefore

$$
y=\frac{1}{x+1+Ce^x}.
$$

In [ ]:
x, C = sp.symbols("x C", real=True)
y = 1/(x+1+C*sp.exp(x))
residual = sp.simplify(sp.diff(y, x) + y - x*y**2)
display(residual)

In [ ]:
x_vals = np.linspace(-2, 3, 700)
for C_value in [-1.5, -0.5, 0, 0.5, 1.5]:
    denom = x_vals+1+C_value*np.exp(x_vals)
    mask = np.abs(denom) > 0.08
    plt.plot(x_vals[mask], 1/denom[mask], label=fr"$C={C_value}$")

plt.ylim(-8, 8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Bernoulli solution family")
plt.legend()
plt.show()

## 4. Reduction using a linear combination

An equation of the form

$$
y'=F(Ax+By+C)
$$

suggests

$$
u=Ax+By+C.
$$

Then

$$
u'=A+By',
$$

which becomes an autonomous and therefore separable equation in $u$.

### Example

Solve

$$
y'=1+(x+y)^2,
\qquad y(0)=0.
$$

Let

$$
u=x+y.
$$

Then

$$
u'=1+y'=2+u^2.
$$

Separate:

$$
\frac{du}{u^2+2}=dx.
$$

Using $u(0)=0$,

$$
u=\sqrt{2}\tan(\sqrt{2}x).
$$

Thus

$$
y=\sqrt{2}\tan(\sqrt{2}x)-x.
$$

In [ ]:
x_vals = np.linspace(-1.05, 1.05, 600)
y_vals = np.sqrt(2)*np.tan(np.sqrt(2)*x_vals)-x_vals

plt.plot(x_vals, y_vals, linewidth=2)
plt.scatter([0], [0], s=70)
plt.axvline(-np.pi/(2*np.sqrt(2)), linestyle="--")
plt.axvline(np.pi/(2*np.sqrt(2)), linestyle="--",
            label="nearest singular points")
plt.ylim(-10, 10)
plt.legend()
plt.show()

## 5. Method-selection checklist

| Structure | Useful method |
|---|---|
| $y'=g(x)h(y)$ | separation |
| $y'+P(x)y=Q(x)$ | integrating factor |
| $M\,dx+N\,dy=0$, $M_y=N_x$ | exact equation |
| $y'=F(y/x)$ | $y=ux$ |
| $y'+Py=Qy^n$ | Bernoulli substitution |
| $y'=F(Ax+By+C)$ | $u=Ax+By+C$ |

Some equations fit more than one category.

## Interactive exploration — Bernoulli solution family and singularities

For

$$
y'+y=xy^2,
$$

the family

$$
y=\frac{1}{x+1+Ce^x}
$$

may develop a vertical asymptote when its denominator becomes zero.

In [ ]:
def bernoulli_family(C=0.5):
    x = np.linspace(-3, 3, 1200)
    denominator = x+1+C*np.exp(x)
    y = np.where(np.abs(denominator) > 0.04, 1/denominator, np.nan)

    plt.plot(x, y, linewidth=2)
    plt.axhline(0, linestyle="--")
    plt.ylim(-10, 10)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(fr"$C={C:.2f}$ in $y=1/(x+1+Ce^x)$")
    plt.show()

    sign_change = np.where(np.sign(denominator[:-1]) != np.sign(denominator[1:]))[0]
    if len(sign_change):
        print("A denominator zero occurs near x =", x[sign_change[0]])
    else:
        print("No denominator zero was detected on the displayed interval.")

if WIDGETS_AVAILABLE:
    interact(
        bernoulli_family,
        C=FloatSlider(min=-2, max=2, step=0.1, value=0.5)
    )
else:
    bernoulli_family()

## Classroom Checkpoint — Exit Check

Which substitution is appropriate?

$$
y'=\left(3x-2y+1\right)^3.
$$

> Pause here. Let students commit to an answer before running the next cell.